# 第 20 课｜RoPE 旋转位置编码与 KV cache

**从“给输入加位置向量”，走到“在每层注意力中旋转 Q、K”。**

本课目标是实现并检查四件事：二维旋转；多频率 RoPE；相对位置点积关系；缓存前后的计算一致性。然后把实现放进一个可训练的小型 GPT。

前置知识：第 15 课的位置、LayerNorm、残差；第 16–17 课的多头注意力与 GPT；第 19 课的 KV cache。只需要你已有的 **PyTorch 环境**，CPU 即可，不需要下载模型或数据。

**学习顺序**：第 1–5 节先做数学与小张量实验；第 6–10 节连接模型、缓存和训练；最后做三项独立练习。已有环境不必重新安装；缺少依赖时在本 Notebook 的一个临时单元运行 `%pip install torch`，然后确认当前内核能 `import torch`。

这份 `.ipynb` 自包含，不读取同名 `.py`；配套脚本也自包含。没有完整学习包或进度 README。

## 使用与实验范围

从上到下执行代码；每次改动后重新执行相关单元。默认用 CPU、单线程、固定种子训练 40 轮小模板模型。输出写到 **当前工作目录** 下的 `lesson_20_outputs`，同名生成文件会覆盖。

本课使用**固定频率、相邻坐标配对、每头全部维度旋转**。保留 LayerNorm、GELU、普通多头注意力，不声称复刻 Llama。完整模型使用 FP32/FP64；缓存只接受等长无 padding 的输入。半精度只用于旋转函数的独立实验。

**第 17–19 课的绝对位置检查点不能直接加载。** 本课从头训练自己的 RoPE 模型，或者加载本课生成的检查点。换一种位置机制，不是删除一个权重键、设置 `strict=False` 就完成了模型转换。

验证集参与模型选择，没有最终独立测试集。模板续写成功不代表通用语言能力，也不用于证明 RoPE 比上一课的位置方案更强。

In [1]:
from __future__ import annotations
import argparse
from dataclasses import asdict, dataclass
import hashlib, json, math, platform, random, tempfile
from pathlib import Path
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# %% 01. 环境与配置
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100
CHECKPOINT_FAMILY = "lesson20_rope_adjacent_v1"


def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")


def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)


def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def nonnegative_int(value: int, name: str) -> None:
    if type(value) is not int or value < 0:
        raise ValueError(f"{name} 必须是非负整数，不接受布尔值")


def check_base(base: float) -> None:
    if (isinstance(base, bool) or not isinstance(base, (int, float))
            or not math.isfinite(base) or base <= 1):
        raise ValueError("本课 rope_base 必须是大于 1 的有限数")


@dataclass(frozen=True)
class RopeConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.1
    rope_base: float = 10000.0

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ff_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= 4:
            raise ValueError("词表需要四个特殊 token 及普通字符")
        if self.d_model % self.n_heads or (self.d_model // self.n_heads) % 2:
            raise ValueError("D 必须整除 H，且每头维度 d_head 必须是偶数")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")
        check_base(self.rope_base)

In [2]:
DEVICE_REQUEST = "cpu"  # 已配好 CUDA 后可改为 "auto"
EPOCHS = 40
SEED = 17
CHECKPOINT = None  # 例如 Path("lesson_20_outputs/rope_model.pt")；只接受第20课检查点
OUTPUT_DIR = Path("lesson_20_outputs")

torch.set_num_threads(1)
device = choose_device(DEVICE_REQUEST)
set_seed(SEED)
print("PyTorch:", torch.__version__, "| device:", device)
print("当前工作目录:", Path.cwd())

PyTorch: 2.10.0+cpu | device: cpu
当前工作目录: /mnt/data/_lesson20_validation/notebook_run


## 1. 位置究竟加在哪里？

第 15–19 课使用：

\[
X_t=E[\text{token}_t]+P[t]
\]

这一课不再添加可学习绝对位置表。对每一层、每个头，先计算投影，再按位置旋转：

\[
q_m=W_Q h_m,\quad k_n=W_K h_n,\quad
\widetilde q_m=R(m)q_m,\quad \widetilde k_n=R(n)k_n
\]

然后用旋转后的 Q/K 计算注意力，**V 保持投影后的值，不旋转**。Q/K 的形状不变。

RoPE 和“把正弦向量相加”都可能用到 sin/cos，但运算位置与方式不同。标准旋转构造见 [RoFormer 原论文 §3.2](https://arxiv.org/html/2104.09864v5)；Q/K 旋转后再写入缓存的实际例子见 [Meta Llama 3 官方实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)。

本文公式把向量写作列向量，代码则用末维坐标计算，避免记号上的转置混淆。

## 2. 二维旋转：先不要急着记长公式

对一个二维向量 \((a,b)\)，逆时针旋转角度 \(\phi\)：

\[
a'=a\cos\phi-b\sin\phi,\qquad
b'=a\sin\phi+b\cos\phi.
\]

角度使用**弧度**：\(90^\circ=\pi/2\)，而不是把数字 90 直接交给 `sin`。

先预测：`[1,0]` 旋转 90° 后是什么？旋转是否改变向量长度？

In [3]:
def rotate_2d(x: torch.Tensor, angle: float) -> torch.Tensor:
    """把最后两维坐标视为一个二维向量；angle 单位是弧度，不是角度。"""
    if not isinstance(x, torch.Tensor) or not x.is_floating_point() or x.shape[-1:] != (2,):
        raise ValueError("x 的末维必须是 2，且为浮点张量")
    if not isinstance(angle, (float, int)) or isinstance(angle, bool) or not math.isfinite(angle):
        raise ValueError("angle 必须是有限弧度值")
    a, b = x.unbind(dim=-1)
    c, s = math.cos(angle), math.sin(angle)
    return torch.stack((a*c - b*s, a*s + b*c), dim=-1)

In [4]:
point = torch.tensor([1.0, 0.0], dtype=torch.float64)
turned = rotate_2d(point, math.pi / 2)
restored = rotate_2d(turned, -math.pi / 2)
print("旋转前:", point.tolist())
print("旋转后:", turned.tolist())
print("原长度/新长度:", point.norm().item(), turned.norm().item())
print("逆向转回:", restored.tolist())

旋转前: [1.0, 0.0]
旋转后: [6.123233995736766e-17, 1.0]
原长度/新长度: 1.0 1.0
逆向转回: [1.0, 0.0]


`6e-17` 这样的极小数可以是浮点计算中理论零的近似，不要用 `== 0` 处理所有浮点检验。

二维旋转矩阵是正交矩阵，因此 \(R(\phi)^T R(\phi)=I\)，保持向量范数。**保持范数不等于保持所有点积**：两个向量分别转不同角度时，相互夹角会改变。后面的多维 RoPE 就由多个这样的二维旋转组成。

## 3. 一对坐标一个频率

假设每头维度为 \(d\)，且 \(d\) 为偶数。本课配对方式为：

```text
(x0,x1), (x2,x3), (x4,x5), ...
```

第 \(i\) 对的频率、位置 \(p\) 对应的旋转角分别是：

\[
\omega_i=\text{base}^{-2i/d},\qquad \phi_{p,i}=p\omega_i,
\quad i=0,\ldots,d/2-1.
\]

本课取 `base=10000`。当 `d=8` 时应得到 `[1, 0.1, 0.01, 0.001]`。这是教学配置，不是所有主流模型共用的固定常数；具体模型需要检查其配置与实现。

注意分母是 **每头维度**，不是模型总隐藏维度。频率固定不需要训练；Q/K 投影矩阵依然可以通过梯度学习。[原论文公式](https://arxiv.org/html/2104.09864v5#S3.SS2.SSS2)

In [5]:
def rope_frequencies(head_dim: int, base: float = 10000.0, *,
                     device: torch.device | str = "cpu", dtype=torch.float32) -> torch.Tensor:
    """omega_i=base**(-2*i/head_dim)，i 从 0 开始；不是 head_dim**(-2*i/base)。"""
    positive_int(head_dim, "head_dim")
    if head_dim % 2:
        raise ValueError("head_dim 必须是偶数")
    check_base(base)
    if dtype not in (torch.float32, torch.float64):
        raise ValueError("频率和相位使用 float32 或 float64，不直接使用半精度")
    exponent = torch.arange(0, head_dim, 2, device=device, dtype=dtype) / head_dim
    return float(base) ** (-exponent)

In [6]:
freq = rope_frequencies(8, dtype=torch.float64)
positions = torch.arange(4, dtype=torch.long)
angles = positions[:, None].double() * freq[None, :]
print("频率:", freq.tolist())
print("角度表 shape:", angles.shape)
print(angles)
print("位置3，第二对坐标的角度:", angles[3, 1].item(), "弧度")

频率: [1.0, 0.1, 0.01, 0.001]
角度表 shape: torch.Size([4, 4])
tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.0000e-01, 1.0000e-02, 1.0000e-03],
        [2.0000e+00, 2.0000e-01, 2.0000e-02, 2.0000e-03],
        [3.0000e+00, 3.0000e-01, 3.0000e-02, 3.0000e-03]], dtype=torch.float64)
位置3，第二对坐标的角度: 0.30000000000000004 弧度


### 3.1 从 `[B,H,T,d]` 到实际代码

`x[...,0::2]` 取每一对的第一个坐标，`x[...,1::2]` 取第二个。它们都是 `[B,H,T,d/2]`。

位置角表为 `[T,d/2]`，前面增加两个长度为 1 的轴，得到 `[1,1,T,d/2]`，便于沿 batch 与 head 广播。

`stack(..., dim=-1)` 把两个新坐标放回一对，最后展平为原来的 `d` 维。

本课数学函数允许负位置，便于测试逆旋转；模型入口仍要求非负起始位置。各 batch 共享一个 `[T]` 位置序列，不涵盖左 padding 或每条请求不同起点。

In [7]:
def _rope_inputs(x: torch.Tensor, positions: torch.Tensor, base: float) -> torch.dtype:
    if (not isinstance(x, torch.Tensor) or x.ndim != 4 or min(x.shape) <= 0
            or x.dtype not in (torch.float16, torch.bfloat16, torch.float32, torch.float64)
            or x.shape[-1] % 2):
        raise ValueError("x 必须是非空浮点 [B,H,T,d]，且 d 是偶数")
    if (not isinstance(positions, torch.Tensor) or positions.dtype != torch.long
            or positions.ndim != 1 or positions.shape[0] != x.shape[-2]
            or positions.device != x.device):
        raise ValueError("positions 必须是同设备、长度为 T 的 torch.long 向量")
    check_base(base)
    # 允许负位置：用于数学逆旋转实验；模型接口另外要求 offset 非负。
    return torch.float64 if x.dtype == torch.float64 else torch.float32


def apply_rope(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """相邻配对 (0,1),(2,3),...；返回相同 shape/dtype/device；不原地修改 x。

    输入 [B,H,T,d]，位置 [T]：本课各 batch 共享同一位置序列。
    半精度输入先升为 FP32 计算相位和旋转，再转回；主模型不启用 autocast。
    """
    compute_dtype = _rope_inputs(x, positions, base)
    freq = rope_frequencies(x.shape[-1], base, device=x.device, dtype=compute_dtype)
    angles = positions.to(compute_dtype)[:, None] * freq[None, :]
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    work = x.to(compute_dtype)
    even, odd = work[..., 0::2], work[..., 1::2]
    rotated_even = even*cos - odd*sin
    rotated_odd = even*sin + odd*cos
    return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2).to(x.dtype)

In [8]:
sample = torch.arange(1, 25, dtype=torch.float64).reshape(1, 1, 3, 8)
pos = torch.arange(3)
rotated = apply_rope(sample, pos)
print("原shape/新shape:", sample.shape, rotated.shape)
print("位置0未变:", torch.allclose(sample[:, :, 0], rotated[:, :, 0]))
print("原范数:", sample.norm(dim=-1))
print("新范数:", rotated.norm(dim=-1))
print("相位不为零的位置:", rotated[0, 0, 1])

原shape/新shape: torch.Size([1, 1, 3, 8]) torch.Size([1, 1, 3, 8])
位置0未变: True
原范数: tensor([[[14.2829, 35.9444, 58.3438]]], dtype=torch.float64)
新范数: tensor([[[14.2829, 35.9444, 58.3438]]], dtype=torch.float64)
相位不为零的位置: tensor([-3.5520, 12.9763,  9.7470, 13.0382, 12.8594, 14.1293, 14.9840, 16.0150],
       dtype=torch.float64)


### 3.2 为什么半精度下还先升到 FP32？

位置索引必须先保留为整数，再在足够精度下算相位。某些整数在低精度里会变成同一个数，后续再转回 float32 也恢复不了。

本课 FP16/BF16 输入的角度、sin/cos 和旋转都先用 FP32 计算，然后返回原输入 dtype；float64 输入保留 float64。注意 FP32 也不是无限精度，这不是任意长位置的精度保证。参考 [PyTorch 数值精度说明](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)。

In [9]:
integer_positions = torch.tensor([2048, 2049], dtype=torch.long)
print("原整数:", integer_positions.tolist())
print("错误：先转FP16再转FP32:", integer_positions.half().float().tolist())
print("直接转FP32:", integer_positions.float().tolist())

原整数: [2048, 2049]
错误：先转FP16再转FP32: [2048.0, 2048.0]
直接转FP32: [2048.0, 2049.0]


## 4. 为什么出现“相对位置”？

固定内容向量 \(q,k\)，分别在位置 \(m,n\) 旋转：

\[
(R(m)q)^T(R(n)k)
=q^T R(m)^T R(n)k
=q^T R(n-m)k.
\]

中间一步来自每对二维旋转满足 \(R(a)^T=R(-a)\)、\(R(a)R(b)=R(a+b)\)。因此，**位置贡献通过差值 \(n-m\) 进入点积**。

这不表示注意力分数只取决于距离：公式仍然包含内容向量 \(q,k\)，而深层模型里的内容向量也已受前文影响。

我们检查两件事：位置 `(2,5)` 同时改为 `(102,105)`，点积不变；只把第二个位置从 5 改为 6，点积可以变化。[RoFormer 相对位置关系](https://arxiv.org/html/2104.09864v5)

In [10]:
q = torch.tensor([1., 2., 3., 4.], dtype=torch.float64).reshape(1,1,1,4)
k = torch.tensor([-1., 3., 2., .5], dtype=torch.float64).reshape(1,1,1,4)
def rotate_at(x, position):
    return apply_rope(x, torch.tensor([position]))

left = (rotate_at(q,2) * rotate_at(k,5)).sum()
right = (q * rotate_at(k,3)).sum()
shifted = (rotate_at(q,102) * rotate_at(k,105)).sum()
changed = (rotate_at(q,2) * rotate_at(k,6)).sum()
print("旋转后点积:", left.item())
print("用相对位置计算:", right.item())
print("全部平移100:", shifted.item())
print("只改变一个位置:", changed.item())
torch.testing.assert_close(left,right,rtol=1e-11,atol=1e-11)
torch.testing.assert_close(left,shifted,rtol=1e-11,atol=1e-11)

旋转后点积: 2.5358084980065585
用相对位置计算: 2.5358084980065576
全部平移100: 2.53580849800656
只改变一个位置: 8.769325897722528


### 4.1 两个容易过度概括的结论

**“距离越远，任何一对 Q/K 的分数都越低”不成立。** 对二维 `q=k=[1,0]`、频率 1，分数就是 `cos(n-m)`，它会振荡。论文关于远距离性质的分析，不能替代对任意向量作单调性保证。

**RoPE 也不替代因果 mask。** 旋转不会自动禁止读取未来，注意力中仍需显式因果约束。

In [11]:
scores_by_distance = [math.cos(delta) for delta in range(8)]
for delta, value in enumerate(scores_by_distance):
    print(f"相对距离={delta}, 点积={value:+.6f}")
print("距离从3变成4，分数反而上升:", scores_by_distance[4] > scores_by_distance[3])

相对距离=0, 点积=+1.000000
相对距离=1, 点积=+0.540302
相对距离=2, 点积=-0.416147
相对距离=3, 点积=-0.989992
相对距离=4, 点积=-0.653644
相对距离=5, 点积=+0.283662
相对距离=6, 点积=+0.960170
相对距离=7, 点积=+0.753902
距离从3变成4，分数反而上升: True


## 5. 工程核对：两种配对布局与独立参考

相邻布局 `(x0,x1),(x2,x3)` 与前后半区布局 `(x0,x2),(x1,x3)` 都能表示二维旋转，但**同一组未转换的坐标不能直接混用**。

把 `[a0,b0,a1,b1]` 改排为 `[a0,a1,b0,b1]`，再使用前后半区版本，最后排回，才与相邻版本相符。加载已有权重时，必须核对相应的 Q/K 投影坐标顺序，不能随意换函数。

另外，用复数 `(a+ib) * exp(i*angle)` 实现一个独立参考，可以检查前面的符号和坐标交错。你不必精通复分析；它只是同一二维旋转的另一种表示。[Meta 相邻配对复数实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)

In [12]:
def rope_complex_reference(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """独立复数参考：a+ib 乘 exp(i*angle)；核对实数版本，不作为模型依赖。"""
    dtype = _rope_inputs(x, positions, base)
    d = x.shape[-1]
    # 独立计算频率，避免两条路径共同调用一个出错的频率函数。
    freq = torch.tensor([float(base)**(-2*i/d) for i in range(d//2)], device=x.device, dtype=dtype)
    angle = positions.to(dtype)[:, None] * freq[None, :]
    phase = torch.complex(angle.cos(), angle.sin())[None, None]
    paired = x.to(dtype).reshape(*x.shape[:-1], d//2, 2).contiguous()
    z = torch.view_as_complex(paired)
    return torch.view_as_real(z * phase).flatten(-2).to(x.dtype)


def interleaved_to_split(x: torch.Tensor) -> torch.Tensor:
    """[a0,b0,a1,b1] -> [a0,a1,b0,b1]，只是坐标排列。"""
    if x.shape[-1] % 2:
        raise ValueError("末维必须为偶数")
    return torch.cat((x[..., 0::2], x[..., 1::2]), dim=-1)


def split_to_interleaved(x: torch.Tensor) -> torch.Tensor:
    if x.shape[-1] % 2:
        raise ValueError("末维必须为偶数")
    a, b = x.chunk(2, dim=-1)
    return torch.stack((a, b), dim=-1).flatten(-2)


def rope_split_half(x: torch.Tensor, positions: torch.Tensor, base: float = 10000.0) -> torch.Tensor:
    """前后半区配对版本；只有同时变换坐标排列，才与相邻版本等价。"""
    dtype = _rope_inputs(x, positions, base)
    freq = rope_frequencies(x.shape[-1], base, device=x.device, dtype=dtype)
    angle = positions.to(dtype)[:, None] * freq[None, :]
    a, b = x.to(dtype).chunk(2, dim=-1)
    c, s = angle.cos()[None, None], angle.sin()[None, None]
    return torch.cat((a*c-b*s, a*s+b*c), dim=-1).to(x.dtype)

In [13]:
test_x = torch.tensor([1.,2.,3.,4.,5.,6.,7.,8.], dtype=torch.float64).reshape(1,1,1,8)
test_pos = torch.tensor([7])
a = apply_rope(test_x,test_pos)
b = rope_complex_reference(test_x,test_pos)
c = split_to_interleaved(rope_split_half(interleaved_to_split(test_x),test_pos))
wrong = rope_split_half(test_x,test_pos)
print("复数参考最大误差:",(a-b).abs().max().item())
print("正确转换布局最大误差:",(a-c).abs().max().item())
print("直接混用布局最大误差:",(a-wrong).abs().max().item())

复数参考最大误差: 0.0
正确转换布局最大误差: 0.0
直接混用布局最大误差: 4.500412856262569


## 6. 接回多头因果自注意力

新计算顺序：

```text
隐藏状态 -> Q/K/V投影 -> 拆头 -> 旋转Q和K -> QKᵀ/√d
       -> 因果mask -> softmax -> 加权V -> 合头 -> 输出投影
```

仍然保持两个 Pre-LN 残差子层。**移除的是输入侧可学习绝对位置表，不是 token embedding，不是因果 mask，也不是投影权重。**

下面先定义 mask 与缓存，再定义注意力。`RotaryCache` 在每层保存已旋转的 K 和未经旋转的 V，同时记录整段请求的起始位置。

In [14]:
# %% 04. 因果 mask 与缓存结构
def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]


def offset_causal_mask(past_len: int, new_len: int, device: torch.device | str = 'cpu') -> torch.Tensor:
    """返回 [1,1,Q,P+Q]。第 i 个新 query 的绝对位置是 P+i。True=允许。"""
    if type(past_len) is not int or past_len < 0:
        raise ValueError('past_len 必须是非负整数')
    positive_int(new_len, 'new_len')
    query_positions = past_len + torch.arange(new_len, device=device)
    key_positions = torch.arange(past_len + new_len, device=device)
    allowed = key_positions[None, :] <= query_positions[:, None]
    return allowed[None, None, :, :]


LayerKV = tuple[torch.Tensor, torch.Tensor]


@dataclass(frozen=True)
class RotaryCache:
    layers: tuple[LayerKV, ...]
    position_offset: int
    rope_base: float
    owner: int  # 仅在当前进程识别模型对象；不是权重版本号，不是安全凭证。

    @property
    def length(self) -> int:
        return self.layers[0][0].shape[-2]


def cache_payload_bytes(cache: RotaryCache | None) -> int:
    if cache is None:
        return 0
    return sum(t.numel()*t.element_size() for pair in cache.layers for t in pair)

In [15]:
# %% 05. RoPE 注意力与 Pre-LN Block
class RotaryAttention(nn.Module):
    def __init__(self, cfg: RopeConfig) -> None:
        super().__init__()
        self.n_heads, self.d_head = cfg.n_heads, cfg.d_model // cfg.n_heads
        self.base = cfg.rope_base
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

    def project(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        b, t, d = x.shape
        def split(proj):
            return proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        return split(self.q_proj), split(self.k_proj), split(self.v_proj)

    def combine(self, q, k, v, allowed):
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        weights = torch.softmax(scores.masked_fill(~allowed, float("-inf")), dim=-1)
        heads = self.dropout(weights) @ v
        b, h, t, d = heads.shape
        return self.out_proj(heads.transpose(1, 2).contiguous().reshape(b, t, h*d))

    def forward(self, x: torch.Tensor, positions: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        q, k, v = self.project(x)
        q = apply_rope(q, positions, self.base)
        k = apply_rope(k, positions, self.base)
        return self.combine(q, k, v, allowed)  # V 不旋转。

    def forward_cached(self, x: torch.Tensor, positions: torch.Tensor,
                       past: LayerKV | None = None) -> tuple[torch.Tensor, LayerKV]:
        q, k_new, v_new = self.project(x)
        q = apply_rope(q, positions, self.base)
        k_new = apply_rope(k_new, positions, self.base)
        past_len = 0 if past is None else past[0].shape[-2]
        if past is None:
            k_all, v_all = k_new, v_new
        else:
            # past[0] 已按各自原位置旋转，不能对拼接后的 k_all 再旋转！
            k_all = torch.cat((past[0], k_new), dim=-2)
            v_all = torch.cat((past[1], v_new), dim=-2)
        allowed = offset_causal_mask(past_len, x.shape[1], x.device)
        return self.combine(q, k_all, v_all, allowed), (k_all, v_all)


class RotaryBlock(nn.Module):
    def __init__(self, cfg: RopeConfig) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = RotaryAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(nn.Linear(cfg.d_model, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.d_model))
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, positions, allowed):
        x = x + self.dropout(self.attn(self.norm1(x), positions, allowed))
        return x + self.dropout(self.ffn(self.norm2(x)))

## 7. RoPE 与 KV cache：每个位置只旋转一次

已有 `P` 个位置，本次输入 `Q` 个新 token；请求整体起点为 `offset`：

```python
positions = offset + P + torch.arange(Q)
```

正确步骤是：对**新 Q、新 K**应用这些位置；把新 K 接到已经旋转过的旧 K 后面；V 只追加。**不能再次旋转整个 K cache。**

局部因果规则依然是 `j <= P+i`，它与整段共同加上的 `offset` 无关。位置角度和 mask 解决的是两个不同问题。

Meta 的公开实现先调用旋转，再将新 K/V 放入缓存；缓存原理见 [Hugging Face 缓存文档](https://huggingface.co/docs/transformers/cache_explanation)。

本实现记录模型对象、起点和 base，检查形状/设备/精度。但它不能检测你对同一模型对象做的所有权重修改。**改权重、改前缀、改布局或改 RoPE 规则后，必须重新 prefill。**

In [16]:
# %% 06. RoPE GPT：不再把绝对位置向量加到 token embedding 上
class RopeGPT(nn.Module):
    def __init__(self, cfg: RopeConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([RotaryBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def _input_check(self, ids):
        if (not isinstance(ids, torch.Tensor) or ids.ndim != 2 or min(ids.shape) <= 0
                or ids.dtype != torch.long):
            raise ValueError("输入必须是非空 [B,T] torch.long")
        if ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须在同一设备")
        if bool(((ids < 0) | (ids >= self.cfg.vocab_size)).any()):
            raise ValueError("token 编号越界")
        if ids.shape[1] > self.cfg.max_len:
            raise ValueError("超过本模型的教学上下文上限；不自动截断")
        if self.token_embedding.weight.dtype not in (torch.float32, torch.float64):
            raise ValueError("完整教学模型仅支持 FP32/FP64；半精度只用于旋转函数实验")
        if torch.is_autocast_enabled(ids.device.type):
            raise ValueError("本课未实现 autocast 路径")

    def forward(self, input_ids: torch.Tensor, *, position_offset: int = 0) -> torch.Tensor:
        self._input_check(input_ids)
        nonnegative_int(position_offset, "position_offset")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = position_offset + torch.arange(input_ids.shape[1], device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        for block in self.blocks:
            x = block(x, positions, allowed)
        return self.lm_head(self.final_norm(x))

    def _cache_check(self, ids, cache, offset):
        self._input_check(ids)
        if any(m.training for m in self.modules()) or torch.is_grad_enabled():
            raise ValueError("缓存路径只接受 eval + torch.no_grad() 推理")
        if bool((ids == PAD_ID).any()):
            raise ValueError("缓存路径不接受 padding；不同长度提示请分开处理")
        if offset is not None:
            nonnegative_int(offset, "position_offset")
        if cache is None:
            return 0, (0 if offset is None else offset)
        if (not isinstance(cache, RotaryCache) or not isinstance(cache.layers, tuple)
                or len(cache.layers) != self.cfg.n_layers):
            raise ValueError("缓存必须是 RotaryCache，且层数与模型一致")
        if cache.owner != id(self) or cache.rope_base != self.cfg.rope_base:
            raise ValueError("缓存所属模型或 RoPE 配置不一致，请重新 prefill")
        nonnegative_int(cache.position_offset, "cache.position_offset")
        if offset is not None and offset != cache.position_offset:
            raise ValueError("不能中途改变缓存的起始位置；请重新 prefill")
        lengths = []
        for pair in cache.layers:
            if not isinstance(pair, tuple) or len(pair) != 2:
                raise ValueError("每层缓存必须是 (K,V)")
            k, v = pair
            if (not isinstance(k, torch.Tensor) or not isinstance(v, torch.Tensor)
                    or k.ndim != 4 or k.shape != v.shape or k.shape[0] != ids.shape[0]
                    or k.shape[1] != self.cfg.n_heads or k.shape[-1] != self.cfg.d_model // self.cfg.n_heads
                    or k.shape[-2] <= 0):
                raise ValueError("缓存需要同形状 [B,H,P,d_head]，P>0")
            for t in pair:
                if (t.device != ids.device or t.dtype != self.token_embedding.weight.dtype
                        or t.requires_grad):
                    raise ValueError("缓存 device/dtype/求导状态不匹配")
            lengths.append(k.shape[-2])
        if len(set(lengths)) != 1:
            raise ValueError("各层历史长度必须一致")
        return lengths[0], cache.position_offset

    def forward_cached(self, input_ids: torch.Tensor, cache: RotaryCache | None = None, *,
                       position_offset: int | None = None) -> tuple[torch.Tensor, RotaryCache]:
        past_len, offset = self._cache_check(input_ids, cache, position_offset)
        new_len = input_ids.shape[1]
        if past_len + new_len > self.cfg.max_len:
            raise ValueError("历史加新增超过上下文；本课不滑动窗口")
        positions = offset + past_len + torch.arange(new_len, device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids))
        updated = []
        for index, block in enumerate(self.blocks):
            past = None if cache is None else cache.layers[index]
            delta, kv = block.attn.forward_cached(block.norm1(x), positions, past)
            x = x + block.dropout(delta)
            x = x + block.dropout(block.ffn(block.norm2(x)))
            updated.append(kv)
        new_cache = RotaryCache(tuple(updated), offset, self.cfg.rope_base, id(self))
        return self.lm_head(self.final_norm(x)), new_cache

In [17]:
probe_cfg = RopeConfig(12,max_len=24,d_model=16,n_heads=2,n_layers=2,ff_dim=32,dropout=0)
probe = RopeGPT(probe_cfg).to(device).eval()
probe_ids = torch.tensor([[1,4,5,6,7]],device=device)
with torch.no_grad():
    first_logits, first_cache = probe.forward_cached(probe_ids[:,:3],position_offset=10)
    second_logits, second_cache = probe.forward_cached(probe_ids[:,3:],first_cache)
    whole_logits = probe(probe_ids,position_offset=10)
    joined = torch.cat((first_logits,second_logits),dim=1)
print("第一次位置: [10,11,12]；第二次位置: [13,14]")
print("原cache长度/新cache长度:",first_cache.length,second_cache.length)
print("第一层K shape:",second_cache.layers[0][0].shape)
print("完整与分块logits最大误差:",(joined-whole_logits).abs().max().item())

第一次位置: [10,11,12]；第二次位置: [13,14]
原cache长度/新cache长度: 3 5
第一层K shape: torch.Size([1, 2, 5, 8])
完整与分块logits最大误差: 2.9802322387695312e-08


### 7.1 为什么整体平移位置不等于“成功外推长上下文”？

对本课固定频率模型，把整段编号从 `0..T-1` 改为 `100..100+T-1`，所有**相对距离不变**。在没有其他绝对位置机制、相同 mask、关闭 Dropout 的前提下，整段 logits 应在浮点误差范围内保持一致。

让序列真正变长会增加相对距离，还会引入模型训练时没有见过的输入与注意力模式。因此，“可以计算很大的位置”或“整体平移无误差”都不能证明模型学会长上下文任务。

[位置插值论文](https://arxiv.org/abs/2306.15595)专门研究了训练长度之外的使用问题；[Transformers RoPE 配置文档](https://huggingface.co/docs/transformers/main/en/internal/rope_utils)区分默认、linear、dynamic、YaRN 等变体。本课只实现固定 base 的默认构造，不把修改 base 宣称为长上下文扩展。

In [18]:
with torch.no_grad():
    start0 = probe(probe_ids,position_offset=0)
    start100 = probe(probe_ids,position_offset=100)
    changed_future = probe_ids.clone()
    changed_future[:,-1] = 10
    future_logits = probe(changed_future)
print("整体平移logits误差:",(start0-start100).abs().max().item())
print("改未来之后的前缀误差:",(start0[:,:-1]-future_logits[:,:-1]).abs().max().item())

整体平移logits误差: 0.0
改未来之后的前缀误差: 0.0


## 8. 数据、损失与训练：已学过的支持代码

仍用 96 条原创模板句，按整句分成 77 条训练和 19 条验证。只在训练侧建词表；标签移位一次；右 padding 的标签使用 `-100`；按有效 token 汇总验证损失。

下面代码可以先折叠，重点回看第 14、17 课的内容。**本次并未控制所有因素对比旧模型，因此训练数值不用于排名两种位置方案。**

In [19]:
# %% 03. 沿用的数据处理：在 Dataset 中只移位一次
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]


def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts


def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]


def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token


def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]


class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]


def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))

In [20]:
# %% 07. 训练与验证：复用已学过的流程，不比较 RoPE 与绝对位置的优劣
def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count


def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}


def evaluate(model: RopeGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)


def train_epoch(model: RopeGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。


def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}


def model_fingerprint(model: nn.Module) -> str:
    digest = hashlib.sha256()
    for name, tensor in model.state_dict().items():
        value = tensor.detach().cpu().contiguous()
        digest.update(name.encode('utf-8'))
        digest.update(str((tuple(value.shape), value.dtype)).encode('utf-8'))
        digest.update(bytes(value.reshape(-1).view(torch.uint8).tolist()))
    return digest.hexdigest()


def train_demo_model(device: torch.device, *, epochs: int = 40, seed: int = 17) -> tuple[RopeGPT, list, dict]:
    positive_int(epochs, "epochs")
    train_texts, val_texts = split_texts(demo_lines(), seed)
    mapping, vocab = build_vocab(train_texts)
    cfg = RopeConfig(len(vocab))
    train_data = CharDataset(train_texts, mapping, cfg.max_len)
    val_data = CharDataset(val_texts, mapping, cfg.max_len)
    train_loader = make_loader(train_data, 16, True, seed)
    train_eval = make_loader(train_data, 16)
    val_loader = make_loader(val_data, 16)
    set_seed(seed)
    model = RopeGPT(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    # 新位置机制，从随机参数重新训练；不加载前课绝对位置权重。
    before = {"train": evaluate(model, train_eval, device), "validation": evaluate(model, val_loader, device)}
    best_loss, best_epoch, best_state = before["validation"]["loss"], 0, snapshot_state(model)
    history = []
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        train_metrics = evaluate(model, train_eval, device)
        validation = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "train": train_metrics, "validation": validation,
                        "online_train": online})
        if validation["loss"] < best_loss:
            best_loss, best_epoch, best_state = validation["loss"], epoch, snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"RoPE 教学模型 epoch={epoch:02d} train={train_metrics['loss']:.4f} val={validation['loss']:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    # 清掉训练残留梯度；比较过程不再创建/更新它们。
    model.zero_grad(set_to_none=True)
    return model, vocab, {"source": "embedded_demo_training", "epochs": epochs, "seed": seed,
                          "train_texts": train_texts, "validation_texts": val_texts,
                          "before": before, "best_epoch": best_epoch, "history": history}

### 8.1 检查点与生成边界

新检查点带有明确的 `family`、模型配置、RoPE base、词表和权重，不含活跃 KV cache。不保存优化器、数据进度和 RNG 状态，所以不是精确续训文件。

旧绝对位置检查点会被拒绝。加载只用本课生成或信任的文件，`weights_only=True` 不是加载任意来源文件的安全保证。

贪心生成支持完整重算和缓存两条路径。每步只取末位置 logits，禁用 BOS/PAD，遇到 EOS、最大生成数或本课上下文上限时停止。缓存路径只输入未处理的新 token；最后返回的 token 可能尚未写入缓存。

In [21]:
# %% 08. 专属检查点：明确拒绝绝对位置检查点

def save_checkpoint(path: Path, model: RopeGPT, vocab: list[str], best_epoch: int) -> None:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size:
        raise ValueError("词表大小与模型不一致")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"family": CHECKPOINT_FAMILY, "config": asdict(model.cfg), "vocab": vocab,
                "model_state": snapshot_state(model), "best_epoch": best_epoch,
                "purpose": "inference_only_not_exact_training_resume"}, path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[RopeGPT, list[str], dict]:
    # weights_only 不是任意不可信文件的安全保证；只加载本课程自行生成/信任的文件。
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("family") != CHECKPOINT_FAMILY:
        raise ValueError("需要第20课 RoPE 检查点；第17–19课绝对位置权重不能直接加载")
    cfg, vocab = RopeConfig(**payload["config"]), payload["vocab"]
    validate_vocab(vocab)
    if cfg.vocab_size != len(vocab):
        raise ValueError("检查点词表大小不匹配")
    with torch.random.fork_rng(devices=[]):
        model = RopeGPT(cfg)
    model.load_state_dict(payload["model_state"], strict=True)
    model.to(device).eval()
    return model, vocab, {"source": "lesson20_checkpoint", "best_epoch": payload["best_epoch"]}

In [22]:
# %% 09. 贪心生成：完整重算与带 RoPE 的 cache
def generate_greedy(model: RopeGPT, vocab: list[str], prompt: str = '', *,
                    use_cache: bool = True, max_new_tokens: int = 24) -> dict:
    validate_vocab(vocab)
    if len(vocab) != model.cfg.vocab_size or not isinstance(prompt, str):
        raise ValueError('词表和模型不一致，或 prompt 不是字符串')
    if type(use_cache) is not bool or type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError('use_cache 必须是布尔值；max_new_tokens 必须是非负整数')
    ids = [BOS_ID] + encode(prompt, {t: i for i, t in enumerate(vocab)})
    if len(ids) > model.cfg.max_len:
        raise ValueError('提示超过上下文；不能静默截断')
    device = next(model.parameters()).device
    current = torch.tensor([ids], device=device, dtype=torch.long)
    cache, trace, new_ids = None, [], []
    reason = 'max_new_tokens' if max_new_tokens == 0 else 'context_limit'
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for step in range(max_new_tokens):
                # 与前课一致：本接口将返回的总长度限制在 max_len，不多吐出第 max_len+1 个编号。
                if current.shape[1] >= model.cfg.max_len:
                    reason = 'context_limit'
                    break
                fed = current if not use_cache or cache is None else current[:, -1:]
                if use_cache:
                    logits, cache = model.forward_cached(fed, cache)
                else:
                    logits = model(fed)
                last = logits[:, -1, :].clone()
                last[:, [BOS_ID, PAD_ID]] = float('-inf')
                next_id = int(last.argmax(-1).item())
                new_ids.append(next_id)
                trace.append({'prediction_step': step, 'input_tokens_this_call': fed.shape[1],
                              'processed_prefix_tokens': current.shape[1],
                              'cache_tokens_after_call': 0 if cache is None else cache.length,
                              'selected_token_id': next_id})
                current = torch.cat((current, current.new_tensor([[next_id]])), dim=1)
                if next_id == EOS_ID:
                    reason = 'eos'
                    break
                reason = 'max_new_tokens'
    finally:
        model.train(was_training)
    visible = ''.join(vocab[i] for i in new_ids if i not in (BOS_ID, EOS_ID, PAD_ID))
    return {'prompt': prompt, 'text': prompt + visible, 'use_cache': use_cache,
            'new_ids': new_ids, 'stop_reason': reason, 'trace': trace,
            'cache_tokens_at_return': 0 if cache is None else cache.length,
            'cache_payload_bytes': cache_payload_bytes(cache),
            'note': '最后选出的 token 尚未送回模型，因此通常未进入 cache；cache 不作为可续接状态返回。'}

## 9. 自动检查与完整实验

自动检查不仅确认“能运行”，还验证数学手算、相对位置恒等式、复数参考输出与梯度、布局变换、未来遮挡、各层缓存和数据边界。

检查中使用 FP64 的几何实验和 FP32 的完整模型实验。浮点核对有明确误差容限，不要求所有设备或运算顺序逐位相等。

先运行检查，再训练。完整实验会检查模型权重在生成前后没有改变，并保存模型、几何结果、训练历史和缓存比较。

In [23]:
# %% 10. 数学和工程实验

def geometry_report() -> dict:
    # 几何验证使用 CPU float64，方便区分代数错误和低精度舍入误差。
    q = torch.tensor([1., 2., 3., 4., -1., 2., .5, -3.], dtype=torch.float64).reshape(1,1,1,8)
    k = torch.tensor([-.5, 1., 2., -1., 3., .5, 2., 1.], dtype=torch.float64).reshape(1,1,1,8)
    def rot(x, p): return apply_rope(x, torch.tensor([p]))
    left = (rot(q, 2) * rot(k, 5)).sum()
    right = (q * rot(k, 3)).sum()
    shifted = (rot(q, 102) * rot(k, 105)).sum()
    relative_changed = (rot(q, 2) * rot(k, 6)).sum()
    pos = torch.tensor([7])
    adjacent = apply_rope(q, pos)
    converted = split_to_interleaved(rope_split_half(interleaved_to_split(q), pos))
    wrong = rope_split_half(q, pos)
    return {
        "frequencies_d8": rope_frequencies(8, dtype=torch.float64).tolist(),
        "rotate_90_degrees": rotate_2d(torch.tensor([1.,0.], dtype=torch.float64), math.pi/2).tolist(),
        "relative_identity_left": left.item(), "relative_identity_right": right.item(),
        "relative_identity_abs_error": abs((left-right).item()),
        "shift_all_positions_abs_error": abs((left-shifted).item()),
        "change_relative_distance_abs_delta": abs((left-relative_changed).item()),
        "norm_max_abs_error": abs((torch.linalg.vector_norm(rot(q,7))-torch.linalg.vector_norm(q)).item()),
        "layout_conversion_error": (adjacent-converted).abs().max().item(),
        "mix_layout_without_conversion_error": (adjacent-wrong).abs().max().item(),
        "nonmonotone_scores_qk_10": [math.cos(i) for i in range(8)],
        "note": "点积仍依赖 q/k 内容。cos 序列说明任意一对向量的分数不保证随距离单调下降。"
    }


def compare_chunked(model: RopeGPT, ids: torch.Tensor, chunks: list[int], *, offset: int = 0) -> dict:
    if not chunks or any(type(c) is not int or c <= 0 for c in chunks) or sum(chunks) != ids.shape[1]:
        raise ValueError("chunks 必须是正整数列表，总和等于 T")
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            full = model(ids, position_offset=offset)
            prefill, full_cache = model.forward_cached(ids, position_offset=offset)
            cache, outputs, start, trace = None, [], 0, []
            for size in chunks:
                logits, cache = model.forward_cached(ids[:,start:start+size], cache, position_offset=offset)
                outputs.append(logits)
                trace.append({"past":start, "new":size, "positions":list(range(offset+start,offset+start+size)),
                              "cache_length":cache.length, "K_shape":list(cache.layers[0][0].shape)})
                start += size
            chunked = torch.cat(outputs, dim=1)
            torch.testing.assert_close(prefill, full, rtol=3e-4, atol=3e-5)
            torch.testing.assert_close(chunked, full, rtol=3e-4, atol=3e-5)
            max_kv = 0.0
            for pair, expected_pair in zip(cache.layers, full_cache.layers):
                for value, expected in zip(pair, expected_pair):
                    torch.testing.assert_close(value, expected, rtol=3e-4, atol=3e-5)
                    max_kv = max(max_kv, (value-expected).abs().max().item())
            shifted = model(ids, position_offset=offset+31)
            torch.testing.assert_close(shifted, full, rtol=3e-4, atol=3e-5)
            return {"chunks":chunks, "position_offset":offset, "trace":trace,
                    "max_logits_error":(chunked-full).abs().max().item(),
                    "max_cache_error":max_kv, "global_shift_logits_error":(shifted-full).abs().max().item(),
                    "cache_bytes":cache_payload_bytes(cache), "rtol":3e-4, "atol":3e-5}
    finally:
        model.train(was_training)

In [24]:
# %% 11. 自动检查：每项带名字，错误立即停止，不依赖可被 -O 移除的 assert
class Checks:
    def __init__(self):
        self.names = []

    def close(self, name, actual, expected, **kwargs):
        torch.testing.assert_close(actual, expected, **kwargs)
        self.names.append(name)

    def true(self, name, condition):
        if not bool(condition):
            raise AssertionError(name)
        self.names.append(name)

    def raises(self, name, fn, expected=ValueError):
        try:
            fn()
        except expected:
            self.names.append(name)
        else:
            raise AssertionError(f"{name}：没有按约定抛出 {expected.__name__}")


def run_checks(device: torch.device = torch.device("cpu")) -> list[str]:
    c = Checks()
    set_seed(200)
    # 1. 独立手算、几何、复数参考、梯度。
    dtype = torch.float64
    x2 = torch.tensor([1.,0.], dtype=dtype, device=device)
    c.close("二维90度旋转", rotate_2d(x2, math.pi/2), x2.new_tensor([0.,1.]), atol=1e-14, rtol=0)
    c.close("二维逆旋转", rotate_2d(rotate_2d(x2, .4), -.4), x2, atol=1e-14, rtol=0)
    c.close("d=8手算频率", rope_frequencies(8, device=device, dtype=dtype),
            x2.new_tensor([1.,.1,.01,.001]), atol=1e-15, rtol=1e-14)
    c.raises("拒绝奇数head维度", lambda: rope_frequencies(3))
    c.raises("拒绝无效base", lambda: rope_frequencies(4, 0.0))
    c.raises("拒绝非有限base", lambda: rope_frequencies(4, float("nan")))
    x = torch.randn(2,3,5,8, device=device, dtype=dtype)
    p = torch.arange(5, device=device, dtype=torch.long)
    before = x.clone()
    y = apply_rope(x,p)
    c.true("旋转保持shape", y.shape == x.shape)
    c.true("旋转保持dtype/device", y.dtype == x.dtype and y.device == x.device)
    c.close("不原地改输入", x, before, rtol=0, atol=0)
    c.close("位置0恒等", y[:,:,0], x[:,:,0], rtol=0, atol=0)
    c.close("保留每个向量范数", y.norm(dim=-1), x.norm(dim=-1), rtol=1e-12, atol=1e-12)
    c.close("负位置逆旋转", apply_rope(y,-p), x, rtol=1e-12, atol=1e-12)
    c.close("组合旋转R(p)R(q)=R(p+q)", apply_rope(y,p+2), apply_rope(x,2*p+2), rtol=1e-12, atol=1e-12)
    c.close("独立复数参考", y, rope_complex_reference(x,p), rtol=1e-12, atol=1e-12)
    q, k = x, torch.randn_like(x)
    qm, kn = apply_rope(q,p+2), apply_rope(k,p+5)
    relative = apply_rope(k, torch.full_like(p,3))
    c.close("相对位置点积恒等式", (qm*kn).sum(-1), (q*relative).sum(-1), rtol=1e-12, atol=1e-12)
    c.close("共同平移不改变点积", (qm*kn).sum(-1),
            (apply_rope(q,p+33)*apply_rope(k,p+36)).sum(-1), rtol=1e-11, atol=1e-11)
    c.true("改相对距离可以改分数", not torch.allclose((qm*kn).sum(-1), (qm*apply_rope(k,p+6)).sum(-1)))
    c.close("布局转换往返", split_to_interleaved(interleaved_to_split(x)), x, rtol=0, atol=0)
    c.close("先转布局再旋转等价", split_to_interleaved(rope_split_half(interleaved_to_split(x),p)), y, rtol=1e-12, atol=1e-12)
    c.true("不转换直接混用不等价", not torch.allclose(rope_split_half(x,p),y))
    noncontiguous = x.transpose(0,1)
    c.true("测试输入确实非连续", not noncontiguous.is_contiguous())
    c.close("支持非连续输入", apply_rope(noncontiguous,p), y.transpose(0,1), rtol=0, atol=0)
    c.raises("拒绝错误position长度", lambda: apply_rope(x,p[:-1]))
    c.raises("拒绝浮点position", lambda: apply_rope(x,p.float()))
    c.raises("拒绝整数特征", lambda: apply_rope(x.long(),p))
    c.raises("拒绝奇数维特征", lambda: apply_rope(x[...,:7],p))
    c.raises("拒绝错误秩", lambda: apply_rope(x[0],p))
    c.raises("拒绝空序列", lambda: apply_rope(x[:,:,:0],p[:0]))
    # 自动求导：独立参考输出与梯度；有限差分只用很小输入。
    xa = x.detach().clone().requires_grad_()
    xb = x.detach().clone().requires_grad_()
    weight = torch.randn_like(x)
    (apply_rope(xa,p)*weight).sum().backward()
    (rope_complex_reference(xb,p)*weight).sum().backward()
    c.close("实数与复数实现梯度一致", xa.grad, xb.grad, rtol=1e-12, atol=1e-12)
    gradx = torch.randn(1,1,2,4, device=device, dtype=dtype, requires_grad=True)
    gp = torch.tensor([2,5], device=device)
    c.true("有限差分gradcheck", torch.autograd.gradcheck(lambda z: apply_rope(z,gp), (gradx,)))
    for low_dtype in (torch.float16, torch.bfloat16):
        low = x.to(low_dtype)
        c.close(f"{low_dtype}内部FP32计算", apply_rope(low,p), apply_rope(low.float(),p).to(low_dtype), rtol=0, atol=0)
    # 2. 模型：因果性、padding、梯度、参数和缓存。
    cfg = RopeConfig(12, max_len=24, d_model=16, n_heads=2, n_layers=2, ff_dim=32, dropout=0)
    c.raises("配置拒绝奇数每头维度", lambda: RopeConfig(12,d_model=12,n_heads=4))
    model = RopeGPT(cfg).to(device).eval()
    ids = torch.tensor([[BOS_ID,4,5,6,7],[BOS_ID,5,6,8,9]], device=device)
    c.true("没有可学习绝对位置表", not any("position_embedding" in n for n,_ in model.named_parameters()))
    c.true("层独立", model.blocks[0].attn.q_proj.weight is not model.blocks[1].attn.q_proj.weight)
    c.true("RoPE没有新增可训练频率参数", not any("freq" in n for n,_ in model.named_parameters()))
    d,f = cfg.d_model,cfg.ff_dim
    expected_params = 2*cfg.vocab_size*d + cfg.n_layers*(4*(d*d+d)+4*d+d*f+f+f*d+d)+2*d
    c.true("参数计数符合公式", count_parameters(model)==expected_params)
    with torch.no_grad():
        full = model(ids)
        c.true("完整输出shape", full.shape==(2,5,12))
        c.true("完整输出有限", torch.isfinite(full).all())
        c.close("整体位置平移后的logits", model(ids,position_offset=23), full, rtol=2e-4, atol=2e-5)
        changed = ids.clone(); changed[:,-1]=10
        c.close("未来不影响前缀", model(changed)[:,:-1], full[:,:-1], rtol=0, atol=0)
        changed = ids.clone(); changed[:,1]=10
        c.true("前文能够影响后续", not torch.allclose(model(changed)[:,-1], full[:,-1], rtol=1e-6, atol=1e-7))
        c.close("截短前缀一致", model(ids[:,:3]), full[:,:3], rtol=2e-4, atol=2e-5)
        padded = torch.cat((ids, torch.full((2,2),PAD_ID,device=device)),dim=1)
        c.close("右padding不改有效输出", model(padded)[:,:5], full, rtol=2e-4, atol=2e-5)
        c.raises("拒绝内部padding", lambda: model(torch.tensor([[1,3,4]], device=device)))
        c.raises("拒绝超长输入", lambda: model(torch.ones(1,25,device=device,dtype=torch.long)))
        c.raises("拒绝负位置偏移", lambda: model(ids,position_offset=-1))
        c.raises("拒绝浮点编号", lambda: model(ids.float()))
        c.raises("拒绝词表外编号", lambda: model(torch.tensor([[99]],device=device)))
        cached_logits, allcache = model.forward_cached(ids)
        c.close("完整prefill一致", cached_logits,full,rtol=0,atol=0)
        c.true("缓存shape", allcache.layers[0][0].shape==(2,2,5,8))
        c.true("缓存字节数", cache_payload_bytes(allcache)==2*2*2*5*16*4)
        # 显式验证第一层缓存的K已旋转，而V保持原始投影。
        first_x = model.blocks[0].norm1(model.token_embedding(ids))
        q0,k0,v0 = model.blocks[0].attn.project(first_x)
        c.close("缓存K=旋转后的K",allcache.layers[0][0],apply_rope(k0,torch.arange(5,device=device)),rtol=0,atol=0)
        c.close("缓存V=未旋转的V",allcache.layers[0][1],v0,rtol=0,atol=0)
        c.true("缓存无求导图",all(not t.requires_grad for pair in allcache.layers for t in pair))
        _, oldcache = model.forward_cached(ids[:,:2],position_offset=7)
        saved_k = oldcache.layers[0][0].clone()
        _, newcache = model.forward_cached(ids[:,2:4],oldcache)
        c.close("追加不改写旧缓存",oldcache.layers[0][0],saved_k,rtol=0,atol=0)
        c.true("保留offset与增长长度",newcache.position_offset==7 and newcache.length==4)
        c.raises("拒绝中途改offset",lambda: model.forward_cached(ids[:,:1],oldcache,position_offset=0))
        c.raises("拒绝错误cache类型",lambda: model.forward_cached(ids[:,:1],tuple()))
        c.raises("拒绝缓存padding",lambda: model.forward_cached(torch.tensor([[1,3]],device=device)))
        other = RopeGPT(cfg).to(device).eval()
        c.raises("拒绝不同模型的缓存",lambda: other.forward_cached(ids[:,:1],oldcache))
        bad = RotaryCache(oldcache.layers,7,500.0,id(model))
        c.raises("拒绝不同base缓存",lambda: model.forward_cached(ids[:,:1],bad))
        short_layers = (oldcache.layers[0],)
        c.raises("拒绝缓存层数不对",lambda: model.forward_cached(ids[:,:1],RotaryCache(short_layers,7,10000.,id(model))))
        wrong_lengths = (oldcache.layers[0], tuple(t[:,:,:1] for t in oldcache.layers[1]))
        c.raises("拒绝各层历史长度不同",lambda: model.forward_cached(ids[:,:1],RotaryCache(wrong_lengths,7,10000.,id(model))))
        wide = torch.ones(2,23,device=device,dtype=torch.long)
        c.raises("拒绝cache历史+新增超长",lambda: model.forward_cached(wide,oldcache))
        # 两个请求交错，不能共享同一个cache变量。
        _, ca = model.forward_cached(ids[:1,:2]); _, cb = model.forward_cached(ids[1:,:2])
        la,ca = model.forward_cached(ids[:1,2:],ca); lb,cb = model.forward_cached(ids[1:,2:],cb)
        c.close("请求A独立cache",la,full[:1,2:],rtol=2e-4,atol=2e-5)
        c.close("请求B独立cache",lb,full[1:,2:],rtol=2e-4,atol=2e-5)
    for chunks,offset in [([1]*5,0),([2,1,2],0),([3,2],11)]:
        result=compare_chunked(model,ids,chunks,offset=offset)
        c.true(f"分块一致{chunks},offset={offset}",result['max_logits_error']<3e-5)
    c.raises("缓存拒绝开启autograd",lambda:model.forward_cached(ids))
    model.train()
    with torch.no_grad():
        c.raises("缓存拒绝训练模式",lambda:model.forward_cached(ids))
    model.eval()
    model.zero_grad(set_to_none=True)
    model(ids).square().mean().backward()
    c.true("全部可训练参数接到有限梯度",all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters()))
    c.true("Q投影非零梯度",model.blocks[0].attn.q_proj.weight.grad.abs().sum()>0)
    c.true("K投影非零梯度",model.blocks[0].attn.k_proj.weight.grad.abs().sum()>0)
    c.true("V投影非零梯度",model.blocks[0].attn.v_proj.weight.grad.abs().sum()>0)
    model.zero_grad(set_to_none=True)
    # 3. 数据契约、生成、保存与加载。
    vocab=SPECIAL_TOKENS+list("甲乙丙丁戊己庚辛")
    mapping={t:i for i,t in enumerate(vocab)}
    dataset=CharDataset(["甲乙","丙"],mapping,24)
    x0,y0=dataset[0]
    c.close("输入目标正确移位",x0,torch.tensor([1,4,5]))
    c.close("目标包含EOS",y0,torch.tensor([4,5,2]))
    batch=collate_batch([dataset[0],dataset[1]])
    c.true("标签padding=-100",batch['targets'][1,-1].item()==IGNORE_INDEX)
    logits=model(batch['input_ids'].to(device))
    summed,n=loss_parts(logits,batch['targets'].to(device))
    c.true("损失仅统计有效token",n==5 and torch.isfinite(summed))
    fingerprint=model_fingerprint(model)
    a=generate_greedy(model,vocab,"甲",use_cache=False,max_new_tokens=5)
    b=generate_greedy(model,vocab,"甲",use_cache=True,max_new_tokens=5)
    c.true("贪心生成编号一致",a['new_ids']==b['new_ids'])
    c.true("停止原因一致",a['stop_reason']==b['stop_reason'])
    c.true("生成不改权重",model_fingerprint(model)==fingerprint)
    c.true("零生成预算",generate_greedy(model,vocab,"甲",max_new_tokens=0)['new_ids']==[])
    model.train(); generate_greedy(model,vocab,"甲",max_new_tokens=2)
    c.true("生成恢复原模式",model.training); model.eval()
    with tempfile.TemporaryDirectory() as temp:
        file=Path(temp)/'model.pt'
        save_checkpoint(file,model,vocab,3)
        loaded,loaded_vocab,meta=load_checkpoint(file,device)
        with torch.no_grad():
            c.close("重载logits一致",loaded(ids),model(ids),rtol=0,atol=0)
        c.true("重载配置词表轮次",loaded.cfg==cfg and loaded_vocab==vocab and meta['best_epoch']==3)
        torch.save({'format_version':1,'config':{},'model_state':{}},file)
        c.raises("明确拒绝前课绝对位置检查点",lambda:load_checkpoint(file,device))
    c.true("缓存不是权重状态",not any('cache' in key for key in model.state_dict()))
    print(f"通过固定检查：{len(c.names)} 项")
    return c.names

In [25]:
check_names = run_checks(device)
print("最后五项检查:",check_names[-5:])

通过固定检查：89 项
最后五项检查: ['生成恢复原模式', '重载logits一致', '重载配置词表轮次', '明确拒绝前课绝对位置检查点', '缓存不是权重状态']


In [26]:
# %% 12. 完整实验与输出

def run_experiment(device: torch.device, output_dir: Path, *, epochs: int = 40,
                   seed: int = 17, checkpoint: Path | None = None) -> dict:
    output_dir=Path(output_dir)
    output_dir.mkdir(parents=True,exist_ok=True)
    geometry=geometry_report()
    if checkpoint is None:
        model,vocab,training=train_demo_model(device,epochs=epochs,seed=seed)
    else:
        model,vocab,training=load_checkpoint(checkpoint,device)
    fingerprint_before=model_fingerprint(model)
    mapping={t:i for i,t in enumerate(vocab)}
    text="我今天正在学习数学。"
    ids=torch.tensor([[BOS_ID]+encode(text,mapping)],device=device)
    if ids.shape[1]>model.cfg.max_len:
        ids=ids[:,:model.cfg.max_len]  # 仅缩短固定探针；生成接口不截断用户提示。
    length=ids.shape[1]
    chunks=[3,2,length-5] if length>5 else [1]*length
    comparisons=[compare_chunked(model,ids,[1]*length),compare_chunked(model,ids,chunks,offset=17)]
    samples=[]
    for prompt in ["我今天正在","小明明天正在","老师今天正在"]:
        if len(prompt)+1>=model.cfg.max_len:
            continue
        a=generate_greedy(model,vocab,prompt,use_cache=False)
        b=generate_greedy(model,vocab,prompt,use_cache=True)
        samples.append({"full":a,"cached":b,"ids_equal":a['new_ids']==b['new_ids'],
                        "stop_equal":a['stop_reason']==b['stop_reason']})
    target=output_dir/'rope_model.pt'
    # 用户把本课原检查点作为输入时，不改写原文件。
    if checkpoint is None or Path(checkpoint).resolve()!=target.resolve():
        save_checkpoint(target,model,vocab,training.get('best_epoch',0))
    loaded,_,_=load_checkpoint(target,device)
    with torch.no_grad():
        torch.testing.assert_close(loaded(ids),model(ids),rtol=0,atol=0)
    fingerprint_after=model_fingerprint(model)
    report={"environment":{"python":platform.python_version(),"torch":str(torch.__version__),
                           "device":str(device),"cuda_available":torch.cuda.is_available(),
                           "cpu_threads":torch.get_num_threads()},
            "config":asdict(model.cfg),"parameters":count_parameters(model),
            "geometry":geometry,"cache_comparisons":comparisons,"generations":samples,
            "weights_unchanged":fingerprint_before==fingerprint_after,"fingerprint":fingerprint_after,
            "limitations":["96条人工模板，验证集参与轮次选择，没有独立测试集",
                           "不与前课绝对位置模型做公平性能对比",
                           "共同平移位置不是增大相对距离，不证明长上下文能力",
                           "固定频率全head RoPE，未实现scaling/GQA/分页缓存",
                           "完整模型FP32/FP64；缓存无padding；修改权重或前缀后重建cache"]}
    for name,content in [('geometry.json',geometry),('training.json',training),('metrics.json',report)]:
        (output_dir/name).write_text(json.dumps(content,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    print(f"可训练参数：{report['parameters']:,}")
    print("数学相对位置误差：",geometry['relative_identity_abs_error'])
    print("逐token最大logits差异：",comparisons[0]['max_logits_error'])
    print("offset=17分块最大logits差异：",comparisons[1]['max_logits_error'])
    print("整体位置平移最大logits差异：",comparisons[0]['global_shift_logits_error'])
    for row in samples:
        print("生成：",row['cached']['text'],"| cache一致：",row['ids_equal'])
    print("结果目录：",output_dir.resolve())
    return report

In [27]:
report = run_experiment(device, OUTPUT_DIR, epochs=EPOCHS, seed=SEED, checkpoint=CHECKPOINT)

RoPE 教学模型 epoch=01 train=2.5943 val=2.6550


RoPE 教学模型 epoch=10 train=0.4221 val=0.4546


RoPE 教学模型 epoch=20 train=0.3970 val=0.4297


RoPE 教学模型 epoch=30 train=0.3907 val=0.4249


RoPE 教学模型 epoch=40 train=0.3882 val=0.4502
可训练参数：105,344
数学相对位置误差： 8.881784197001252e-16
逐token最大logits差异： 2.86102294921875e-06
offset=17分块最大logits差异： 1.9073486328125e-06
整体位置平移最大logits差异： 1.9073486328125e-06
生成： 我今天正在检查数据。 | cache一致： True
生成： 小明明天正在检查数据。 | cache一致： True
生成： 老师今天正在检查数据。 | cache一致： True
结果目录： /mnt/data/_lesson20_validation/notebook_run/lesson_20_outputs


### 9.1 阅读输出，不把模板实验当作能力榜单

`training.json` 记录每轮训练/验证指标。默认配置下模型为两层、隐藏维度 64、四个头、FFN 256；使用默认词表时有 **105,344 个参数**。与前课相同规模的绝对位置版本相比，少的是 64×64 个位置表参数，其他设计仍有独立初始化过程。

`metrics.json` 包括几何关系、逐 token 与分块 logits 误差、各层 K/V 误差、整体位置平移结果、全部演示生成及权重指纹。

**边界**：验证集参与模型选择；没有独立测试集；训练和验证共用人工模板。生成“我今天正在检查数据。”不意味着模型理解了今天、检查或数据的现实含义。

In [28]:
training = json.loads((OUTPUT_DIR / "training.json").read_text(encoding="utf-8"))
print("模型参数:",report["parameters"])
print("最佳验证轮次:",training.get("best_epoch"))
if "history" in training:
    best = next((row for row in training["history"] if row["epoch"]==training["best_epoch"]),None)
    print("训练前验证损失:",training["before"]["validation"]["loss"])
    if best is not None:
        print("所选验证损失:",best["validation"]["loss"])
print("每条生成的cache一致性:",[r["ids_equal"] for r in report["generations"]])
print("参数是否保持不变:",report["weights_unchanged"])

模型参数: 105344
最佳验证轮次: 23
训练前验证损失: 3.722512803636156
所选验证损失: 0.4220386840201713
每条生成的cache一致性: [True, True, True]
参数是否保持不变: True


## 10. 运行完整脚本

在 `.py` 所在目录打开终端：

```bash
python lesson_20_rope_and_kv_cache.py
python lesson_20_rope_and_kv_cache.py --checks-only
python lesson_20_rope_and_kv_cache.py --device auto
```

已有**本课**检查点时跳过训练：

```bash
python lesson_20_rope_and_kv_cache.py --checkpoint lesson_20_outputs/rope_model.pt
```

默认输出：

```text
lesson_20_outputs/
    rope_model.pt
    geometry.json
    training.json
    metrics.json
```

结果写到当前工作目录，终端会打印绝对路径。默认同名输出覆盖；当输入检查点与目标 `rope_model.pt` 为同一个文件时，会保留输入文件，不重新写它。

GPU 路径只有在你已有可用的 CUDA PyTorch 环境时才可使用。本讲义附带的运行输出来自 CPU；不是 RTX 3060 的实测。

## 11. 独立练习：先写，再开检查

运行全部单元成功并不代表完成了练习。下面三个检查开关默认关闭，函数默认抛出 `NotImplementedError`。写好一个，再打开对应开关。

检查函数使用本课独立参考及完整模型，不替你完成练习。

In [29]:
def check_ex_a(fn):
    gen = torch.Generator().manual_seed(208)
    x = torch.randn(2, 3, 4, 8, generator=gen, dtype=torch.float64)
    positions = torch.tensor([0, 2, 5, 9])
    original = x.clone()
    actual = fn(x, positions)
    expected = rope_complex_reference(x, positions)
    torch.testing.assert_close(actual, expected, rtol=1e-11, atol=1e-11)
    torch.testing.assert_close(x, original, rtol=0, atol=0)
    torch.testing.assert_close(actual.norm(dim=-1), x.norm(dim=-1), rtol=1e-11, atol=1e-11)
    torch.testing.assert_close(fn(actual, -positions), x, rtol=1e-11, atol=1e-11)
    torch.testing.assert_close(fn(x.transpose(0,1), positions), expected.transpose(0,1), rtol=1e-11, atol=1e-11)
    xa, xb = x.clone().requires_grad_(), x.clone().requires_grad_()
    weight = torch.randn(x.shape, generator=gen, dtype=torch.float64)
    (fn(xa,positions)*weight).sum().backward()
    (rope_complex_reference(xb,positions)*weight).sum().backward()
    torch.testing.assert_close(xa.grad,xb.grad,rtol=1e-11,atol=1e-11)
    print("练习 A：值、范数、逆旋转、非连续输入、输入保护和梯度通过")


def check_ex_b(fn):
    gen = torch.Generator().manual_seed(209)
    q,k,v = [torch.randn(2,2,5,4,generator=gen,dtype=torch.float64) for _ in range(3)]
    positions = torch.arange(11,16)
    # 参考一次完整计算；使用独立复数实现，不复用练习的增量代码。
    qr,kr = rope_complex_reference(q,positions),rope_complex_reference(k,positions)
    scores = qr @ kr.transpose(-1,-2) / 2.0
    allowed = torch.ones(5,5,dtype=torch.bool).tril()
    expected = scores.masked_fill(~allowed,float('-inf')).softmax(-1) @ v
    actual,cache = fn(q,k,v,positions)
    torch.testing.assert_close(actual,expected,rtol=1e-11,atol=1e-11)
    torch.testing.assert_close(cache[0],kr,rtol=1e-11,atol=1e-11)
    torch.testing.assert_close(cache[1],v,rtol=0,atol=0)
    first,old = fn(q[:,:,:2],k[:,:,:2],v[:,:,:2],positions[:2])
    old_k = old[0].clone()
    second,cache2 = fn(q[:,:,2:],k[:,:,2:],v[:,:,2:],positions[2:],old)
    torch.testing.assert_close(torch.cat((first,second),dim=-2),expected,rtol=1e-11,atol=1e-11)
    torch.testing.assert_close(old[0],old_k,rtol=0,atol=0)
    torch.testing.assert_close(cache2[0],kr,rtol=1e-11,atol=1e-11)
    torch.testing.assert_close(cache2[1],v,rtol=0,atol=0)
    print("练习 B：完整与分块结果、非零位置、已旋转K、未旋转V及旧缓存保护通过")


def check_ex_c(fn):
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(210)
        model = RopeGPT(RopeConfig(12,max_len=16,d_model=16,n_heads=2,n_layers=2,ff_dim=32,dropout=.1))
    ids = torch.tensor([[1,4,5,6,7]])
    model.eval()
    with torch.no_grad():
        expected = model(ids,position_offset=19)
        _,expected_cache = model.forward_cached(ids,position_offset=19)
    before = model_fingerprint(model)
    model.train()
    output,cache = fn(model,ids,[2,1,2],position_offset=19)
    torch.testing.assert_close(output,expected,rtol=3e-4,atol=3e-5)
    if not model.training or output.requires_grad or cache.length != 5:
        raise AssertionError("需要恢复原模式、禁用梯度、保留全部缓存")
    for pair,ref in zip(cache.layers,expected_cache.layers):
        for a,b in zip(pair,ref):
            torch.testing.assert_close(a,b,rtol=3e-4,atol=3e-5)
    if model_fingerprint(model) != before:
        raise AssertionError("练习不得修改模型参数")
    for bad in ([],[0,5],[2,2],[True,4]):
        try:
            fn(model,ids,bad,position_offset=19)
        except ValueError:
            pass
        else:
            raise AssertionError("无效 chunks 应抛出 ValueError")
    print("练习 C：分块logits、cache、模式恢复、不改权重和无效分块检查通过")

### 练习 A：自己实现 `rope_ex()`

输入为 `[B,H,T,d]` 浮点张量、`[T]` long 位置，返回同形状。使用相邻配对、指定 base，不原地修改输入；支持 FP64、负位置和非连续输入，保持自动求导。

可以调用 `_rope_inputs()` 做校验，但**不要直接调用 `apply_rope()`、复数参考或答案函数返回结果**。亲自完成频率、角度、sin/cos、两条旋转公式及坐标还原。

先手算：`d=4，base=10000` 时两个频率是多少？位置 3 的两个角度呢？

In [30]:
def rope_ex(x,positions,base=10000.0):
    raise NotImplementedError("在这里亲自实现相邻坐标RoPE")

RUN_EX_A = False
if RUN_EX_A:
    check_ex_a(rope_ex)

### 练习 B：实现带 cache 的单次多头注意力核心

实现 `head_attention_ex(q_new,k_new,v_new,positions,past=None,base=10000.0)`。

约定：三个新投影均为 `[B,H,Q,d]`；`positions` 是这些新 token 的**真实位置**，可以从非零处开始；`past` 为 `(K_rotated,V_raw)`，两者都是 `[B,H,P,d]`。测试只提供合法输入。

输出 `(heads, new_cache)`：`heads` 为 `[B,H,Q,d]`；缓存为拼接后的 `[B,H,P+Q,d]`。

可以调用 `apply_rope()`，但是要自己完成追加、带偏移的因果 mask、缩放点积、softmax 和 V 汇总。不要旋转 V；不要对旧 K 重复旋转；不要原地改旧缓存；不要把 Q 留在 cache 里。

本练习是数学核心，可以构建计算图；完整模型的持久 cache 接口仍然限定为推理用途。

In [31]:
def head_attention_ex(q_new,k_new,v_new,positions,past=None,base=10000.0):
    raise NotImplementedError("旋转新Q/K，追加cache，按历史长度建立mask，再汇总V")

RUN_EX_B = False
if RUN_EX_B:
    check_ex_b(head_attention_ex)

### 练习 C：分块前向与模式恢复

实现 `chunked_logits_ex(model,ids,chunks,position_offset=0)`，返回拼接的全部 logits 和最终缓存。

要求 `chunks` 为正整数列表且总和等于输入长度，不接受布尔值。进入 `eval()+no_grad()`，只传新增 token，使用同一个 `position_offset` 与不断增长的 cache，最后恢复原训练模式。不能更改权重。

不要用一次完整 `model(ids)` 替代分块；需要真正调用 `forward_cached()`。这不是生成循环，不抽样、不追加新编号，只重放已知输入。

In [32]:
def chunked_logits_ex(model,ids,chunks,position_offset=0):
    raise NotImplementedError("校验分块、传递cache、拼接logits、恢复模式")

RUN_EX_C = False
if RUN_EX_C:
    check_ex_c(chunked_logits_ex)

---
## 12. 参考答案——完成练习后再向下阅读

答案用于对照计算顺序。读完答案以后，换一组位置、head_dim、chunks 再自己验证。下方参考答案测试可以运行，但它们不等于上面的练习已经完成。

In [33]:
def rope_answer(x, positions, base=10000.0):
    dtype = _rope_inputs(x, positions, base)
    d = x.shape[-1]
    frequency = float(base) ** (-torch.arange(0,d,2,device=x.device,dtype=dtype)/d)
    phase = positions.to(dtype)[:,None] * frequency[None,:]
    cos,sin = phase.cos()[None,None],phase.sin()[None,None]
    a,b = x.to(dtype)[...,0::2],x.to(dtype)[...,1::2]
    return torch.stack((a*cos-b*sin,a*sin+b*cos),dim=-1).flatten(-2).to(x.dtype)

In [34]:
check_ex_a(rope_answer)

练习 A：值、范数、逆旋转、非连续输入、输入保护和梯度通过


In [35]:
def head_attention_answer(q_new,k_new,v_new,positions,past=None,base=10000.0):
    q_rot = apply_rope(q_new,positions,base)
    k_rot = apply_rope(k_new,positions,base)
    past_len = 0 if past is None else past[0].shape[-2]
    if past is None:
        k_all,v_all = k_rot,v_new
    else:
        k_all = torch.cat((past[0],k_rot),dim=-2)
        v_all = torch.cat((past[1],v_new),dim=-2)
    q_len,d = q_new.shape[-2:]
    query_index = past_len + torch.arange(q_len,device=q_new.device)
    key_index = torch.arange(past_len+q_len,device=q_new.device)
    allowed = key_index[None,:] <= query_index[:,None]
    scores = q_rot @ k_all.transpose(-1,-2) / math.sqrt(d)
    weights = torch.softmax(scores.masked_fill(~allowed,float('-inf')),dim=-1)
    return weights @ v_all,(k_all,v_all)

In [36]:
check_ex_b(head_attention_answer)

练习 B：完整与分块结果、非零位置、已旋转K、未旋转V及旧缓存保护通过


In [37]:
def chunked_logits_answer(model,ids,chunks,position_offset=0):
    if (not isinstance(chunks,list) or not chunks
            or any(type(n) is not int or n<=0 for n in chunks) or sum(chunks)!=ids.shape[1]):
        raise ValueError("chunks 必须为正整数列表，且总和等于输入长度")
    old_mode = model.training
    try:
        model.eval()
        results,cache,start = [],None,0
        with torch.no_grad():
            for length in chunks:
                part = ids[:,start:start+length]
                logits,cache = model.forward_cached(part,cache,position_offset=position_offset)
                results.append(logits)
                start += length
        return torch.cat(results,dim=1),cache
    finally:
        model.train(old_mode)

In [38]:
check_ex_c(chunked_logits_answer)

练习 C：分块logits、cache、模式恢复、不改权重和无效分块检查通过


## 13. 常见错误与定位方式

| 现象 | 优先检查 |
|---|---|
| 范数变化明显 | sin 的正负号、每对两个坐标的组合 |
| 形状正确但参考不匹配 | 相邻与前后半区布局混用，或频率分母用错 |
| 完整前向正常，第二块开始不同 | 位置漏加历史长度，或旧 K 被重复旋转 |
| 下一块无法读取多数历史 | 直接套左上角非方阵因果 mask |
| 整体平移后logits明显变化 | V 被旋转、仍残留绝对位置表、频率/布局不一致、Dropout 未关闭 |
| 改未来后前缀发生变化 | 因果 mask 漏加，不能靠 RoPE 遮挡未来 |
| `.pt` 加载失败 | 检查点属于第17–19课，或构造配置/词表不匹配 |
| 大位置运行了但任务变差 | 可计算不等于训练过的长距离泛化，不能只调 max_len 得出能力结论 |

**注意**：合法模型可能同时采用其他绝对位置信息，或使用随序列长度变化的缩放机制；本课整体平移检验不是所有真实模型的通用不变量。

验收：能够不看答案写出二维旋转，解释频率和张量轴，推导相对位置点积，独立通过三道练习，并指出实验不能证明的事情。

## 14. 延伸阅读与下一课

1. [RoFormer 原论文](https://arxiv.org/html/2104.09864v5)：先读 §3.2，核对本课公式；推导可以在实现后回看。
2. [Meta Llama 3 模型实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)：查看 `apply_rotary_emb`、注意力中旋转和写缓存的先后顺序。不要把本课其他模块与 Llama 等同。
3. [Hugging Face 缓存说明](https://huggingface.co/docs/transformers/cache_explanation)：复习历史长度、注意力形状与推理 cache。
4. [RoPE 配置与变体](https://huggingface.co/docs/transformers/main/en/internal/rope_utils)：知道配置存在差异即可；本课不实现长上下文缩放。
5. [位置插值论文](https://arxiv.org/abs/2306.15595)：理解可计算的大位置与长上下文能力的区别。
6. [PyTorch 数值精度说明](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html)：为什么要用误差容限，为什么不同计算顺序不必逐位相等。

**下一课：第 21 课——RMSNorm 与 SwiGLU。** 保留本课 RoPE 和缓存流程，继续理解现代解码器的归一化、门控前馈网络、参数量和工程取舍。